In [10]:
import pandas as pd
from fredapi import Fred

fred = Fred(api_key="cfa03dec4e594e0e37455bad49de64a8")

series = {
    # "pmi": "NAPMNOI",
    "cpi": "CPIAUCSL",
    "fedfunds": "FEDFUNDS",
    "industrial_production": "INDPRO",
    "gdp": "GDPC1",
    "retail_sales": "RSAFS",
    "unemployment": "UNRATE",
    "t10y": "DGS10",
    "t2y": "DGS2",
    "t3m": "TB3MS",
    "aaa_yield": "AAA",
    "vix": "VIXCLS",
    "sp500": "SP500"
}

macro = pd.DataFrame()

for name, code in series.items():
    print(f"Downloading {name} ({code})…")
    data = fred.get_series(code)
    data = data.to_frame(name)
    data.index = pd.to_datetime(data.index)
    macro = macro.join(data, how="outer")

# Compute yield spread
macro["yield_spread_10y_2y"] = macro["t10y"] - macro["t2y"]

macro = macro.sort_index()
macro.to_parquet("../data/macro/macro_all_fred.parquet")

macro.head()


,cpi,fedfunds,industrial_production,gdp,retail_sales,unemployment,t10y,t2y,t3m,aaa_yield,vix,sp500,yield_spread_10y_2y
1919-01-01,NaN,NaN,4.8654,NaN,NaN,NaN,NaN,NaN,NaN,5.35,NaN,NaN,NaN
1919-02-01,NaN,NaN,4.6504,NaN,NaN,NaN,NaN,NaN,NaN,5.35,NaN,NaN,NaN
1919-03-01,NaN,NaN,4.5160,NaN,NaN,NaN,NaN,NaN,NaN,5.39,NaN,NaN,NaN
1919-04-01,NaN,NaN,4.5966,NaN,NaN,NaN,NaN,NaN,NaN,5.44,NaN,NaN,NaN
1919-05-01,NaN,NaN,4.6235,NaN,NaN,NaN,NaN,NaN,NaN,5.39,NaN,NaN,NaN


In [12]:
# Keep only 2014–2024
macro = macro.loc["2014-01-01":"2024-12-31"]

# Convert to daily frequency (because some series are monthly/quarterly)
macro = macro.asfreq("D").ffill()

# Save cleaned version
macro.to_parquet("../data/model/macro_features.parquet")

macro.head(), macro.tail()

(                cpi  fedfunds  industrial_production        gdp  retail_sales  \
 2014-01-01  235.288      0.07                99.9899  17953.974      411561.0   
 2014-01-02  235.288      0.07                99.9899  17953.974      411561.0   
 2014-01-03  235.288      0.07                99.9899  17953.974      411561.0   
 2014-01-04  235.288      0.07                99.9899  17953.974      411561.0   
 2014-01-05  235.288      0.07                99.9899  17953.974      411561.0   
 
             unemployment  t10y   t2y   t3m  aaa_yield    vix  sp500  \
 2014-01-01           6.6   NaN   NaN  0.04       4.49    NaN    NaN   
 2014-01-02           6.6  3.00  0.39  0.04       4.49  14.23    NaN   
 2014-01-03           6.6  3.01  0.41  0.04       4.49  13.76    NaN   
 2014-01-04           6.6  3.01  0.41  0.04       4.49  13.76    NaN   
 2014-01-05           6.6  3.01  0.41  0.04       4.49  13.76    NaN   
 
             yield_spread_10y_2y  
 2014-01-01                  NaN  
 2